# Experimento — Projeto_Exemplo_MLP_2_classes (Luciano)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpeuzinCosta/kws-borda/blob/main/notebooks/experimentos/Projeto_Exemplo_MLP_2_classes.ipynb)

Exploração do GSC v2 e MLP mínima em **2 classes** (*yes*/*no*).

Notebook autônomo: baixa o GSC v2 sozinho em `/content/data` e não depende da bancada do repositório.

In [ ]:
import random
from pathlib import Path

import torch
import torchaudio
import torch.nn.functional as F

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchaudio.datasets import SPEECHCOMMANDS
from torchaudio.transforms import MFCC

import os

In [ ]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = Path("/content/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
AUDIO_LENGTH = SAMPLE_RATE  # 1 segundo

COMMANDS = [
    "yes",
    "no",
]

label_to_idx = {
    label: index
    for index, label in enumerate(COMMANDS)
}
print(label_to_idx)

{'yes': 0, 'no': 1}


In [ ]:
if not os.path.exists("/content/data/SpeechCommands/speech_commands_v0.02"):
    dataset = SPEECHCOMMANDS(
        root=DATA_ROOT,
        url="speech_commands_v0.02",
        download=True
    )

print("Número de áudios:", len(dataset))

Número de áudios: 105829


In [ ]:
waveform, sample_rate, label, speaker_id, utterance_number = dataset[0]

print("Formato do áudio:", waveform.shape)
print("Taxa de amostragem:", sample_rate)
print("Palavra:", label)
print("Identificador do locutor:", speaker_id)
print("Número da gravação:", utterance_number)

Formato do áudio: torch.Size([1, 16000])
Taxa de amostragem: 16000
Palavra: backward
Identificador do locutor: 0165e0e8
Número da gravação: 0


In [ ]:
# Valores brutos do dataset são normalizados via escala PCM [-1, 1], eles são uma representação digital dos valores analógicos de tensão captados pelo microfone
# Taxa de amostragem é 16kHz

print(waveform)
print(waveform[0, :20])

tensor([[-0.0658, -0.0709, -0.0753,  ..., -0.0700, -0.0731, -0.0704]])
tensor([-0.0658, -0.0709, -0.0753, -0.0793, -0.0806, -0.0818, -0.0770, -0.0732,
        -0.0691, -0.0649, -0.0664, -0.0669, -0.0731, -0.0786, -0.0852, -0.0891,
        -0.0915, -0.0918, -0.0917, -0.0902])


In [ ]:
for index in range(5):

    waveform, sample_rate, label, speaker_id, utterance_number = dataset[index]

    print(
        index,
        waveform.shape,
        sample_rate,
        label,
        speaker_id,
        utterance_number
    )

0 torch.Size([1, 16000]) 16000 backward 0165e0e8 0
1 torch.Size([1, 16000]) 16000 backward 017c4098 0
2 torch.Size([1, 16000]) 16000 backward 017c4098 1
3 torch.Size([1, 16000]) 16000 backward 017c4098 2
4 torch.Size([1, 16000]) 16000 backward 017c4098 3


In [ ]:
from collections import Counter

labels = []

for index in range(len(dataset)):
    _, _, label, _, _ = dataset.get_metadata(index)
    labels.append(label)

class_counts = Counter(labels)

print("Número de classes:", len(class_counts))
print(class_counts)

Número de classes: 35
Counter({'five': 4052, 'zero': 4052, 'yes': 4044, 'seven': 3998, 'no': 3941, 'nine': 3934, 'down': 3917, 'one': 3890, 'go': 3880, 'two': 3880, 'stop': 3872, 'six': 3860, 'on': 3845, 'left': 3801, 'eight': 3787, 'right': 3778, 'off': 3745, 'four': 3728, 'three': 3727, 'up': 3723, 'dog': 2128, 'wow': 2123, 'house': 2113, 'marvin': 2100, 'bird': 2064, 'happy': 2054, 'cat': 2031, 'sheila': 2022, 'bed': 2014, 'tree': 1759, 'backward': 1664, 'visual': 1592, 'follow': 1579, 'learn': 1575, 'forward': 1557})


In [ ]:
duration = waveform.shape[1] / sample_rate

print("Duração:", duration, "segundos")

Duração: 1.0 segundos


tensor([[-0.0658, -0.0709, -0.0753,  ..., -0.0700, -0.0731, -0.0704]])

In [ ]:
dataset.get_metadata(5)

('speech_commands_v0.02/backward/017c4098_nohash_4.wav',
 16000,
 'backward',
 '017c4098',
 4)

In [ ]:
class KWSDataset(Dataset):

    def __init__(self, root, subset):

        self.dataset = SPEECHCOMMANDS(
            root=root, # Caminho da pasta do dataset
            url="speech_commands_v0.02", # Versão do dataset
            subset=subset, # Podemos escolher entre treino, val e teste.
            download=False # Não vai fazer download, só carregar na variável
        )

        self.indices = []

        for i in range(len(self.dataset)):
            _, _, label, _, _ = self.dataset.get_metadata(i) # Pega o rótulo de cada amostra do dataset

            if label in COMMANDS: # Filtra pra deixar apenas os rótulos selecionados para o trabalho
                self.indices.append(i) #

        self.mfcc = MFCC(
            sample_rate=16000,
            n_mfcc=40,
            melkwargs={
                "n_fft": 400,
                "hop_length": 160,
                "n_mels": 40,
                "center": False
            }
        )

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):

        original_index = self.indices[i]

        waveform, _, label, _, _ = self.dataset[original_index]

        # Corta para no máximo 16.000 amostras, caso a amostra tenha mais de 16k amostras
        waveform = waveform[:, :16000]

        # Completa com zeros quando, caso a amostra tenha mais de 16k amostras
        if waveform.shape[1] < 16000:
            waveform = F.pad(
                waveform,
                (0, 16000 - waveform.shape[1])
            )

        features = self.mfcc(waveform) # As features ficam com formato 40x98, cada coluna uma janela temporal, cada linha um componente MFCC

        mean = features.mean(dim=2, keepdim=True)
        std = features.std(dim=2, keepdim=True)

        features = (features - mean) / (std + 1e-6) # normalização

        target = label_to_idx[label]

        return features, target

In [ ]:
class KWSDataLoaders:

    def __init__(self, root, batch_size=256):

        self.batch_size = batch_size

        self.train_dataset = KWSDataset(
            root=root,
            subset="training"
        )

        self.val_dataset = KWSDataset(
            root=root,
            subset="validation"
        )

        self.test_dataset = KWSDataset(
            root=root,
            subset="testing"
        )

    def train_dataloader(self):

        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )

    def val_dataloader(self):

        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

    def test_dataloader(self):

        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False
        )

In [ ]:
data = KWSDataLoaders(
    root=DATA_ROOT,
    batch_size=256
)

train_loader = data.train_dataloader()
val_loader = data.val_dataloader()
test_loader = data.test_dataloader()

In [ ]:
features, targets = next(iter(train_loader))

print(features.shape)
print(targets.shape)

torch.Size([256, 1, 40, 98])
torch.Size([256])


In [ ]:
features

tensor([[[[-1.1434, -1.1331, -1.1423,  ..., -1.0140, -0.9338, -0.9553],
          [ 0.4746,  0.4907,  0.4896,  ...,  0.5304,  0.7019,  0.5055],
          [ 0.0176,  0.3872,  0.1204,  ...,  0.5856,  0.2293, -0.1402],
          ...,
          [-1.3114, -0.6600, -1.5027,  ..., -0.3055, -1.1993,  0.5625],
          [-0.7006, -0.4308,  0.7562,  ..., -0.3883,  0.7537,  0.3536],
          [-1.1930, -0.4280,  0.4593,  ...,  0.6278, -0.5483, -0.4257]]],


        [[[ 0.0421, -0.0542, -0.2077,  ..., -0.9743, -0.9756, -0.8576],
          [-0.3927, -0.6502, -0.5933,  ...,  0.8232,  0.8137,  0.9040],
          [ 0.3301,  0.4540,  0.4876,  ...,  0.1233,  0.1095,  0.2814],
          ...,
          [ 0.6332,  1.6053,  0.4749,  ..., -0.8486, -1.0264, -0.4302],
          [ 1.0653,  2.0156,  0.2887,  ..., -0.1176, -0.4286, -0.1393],
          [ 0.6889,  0.0257,  1.5019,  ...,  0.3266, -0.0989, -2.1880]]],


        [[[-1.0751, -0.9390, -0.8235,  ..., -1.0446, -1.0996, -1.0807],
          [-1.0885, -1.154

In [ ]:
len(train_loader)

25

In [ ]:
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(40 * 98, 32),
            nn.ReLU(),

            #nn.Linear(128, 64),
            #nn.ReLU(),

            nn.Linear(32, len(COMMANDS))
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = MLP().to(device)

loss_function = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.005
)

number_epochs = 3

In [ ]:
for epoch in range(number_epochs):

    # Treinamento
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for batch_index, (features, targets) in enumerate(
        train_loader,
        start=1
    ):

        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(features)

        loss = loss_function(
            logits,
            targets
        )

        loss.backward()
        optimizer.step()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        batch_correct = (
            predictions == targets
        ).sum().item()

        batch_accuracy = (
            batch_correct / targets.size(0)
        )

        train_loss += loss.item() * targets.size(0)
        train_correct += batch_correct
        train_total += targets.size(0)

        print(
            f"Época {epoch + 1:02d} | "
            f"Treino | "
            f"Batch {batch_index}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f} | "
            f"Acurácia: {batch_accuracy:.4f}"
        )

    train_loss = train_loss / train_total
    train_accuracy = train_correct / train_total

    # Validação
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for batch_index, (features, targets) in enumerate(
            val_loader,
            start=1
        ):

            features = features.to(device)
            targets = targets.to(device)

            logits = model(features)

            loss = loss_function(
                logits,
                targets
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            batch_correct = (
                predictions == targets
            ).sum().item()

            batch_accuracy = (
                batch_correct / targets.size(0)
            )

            val_loss += loss.item() * targets.size(0)
            val_correct += batch_correct
            val_total += targets.size(0)

            print(
                f"Época {epoch + 1:02d} | "
                f"Validação | "
                f"Batch {batch_index}/{len(val_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Acurácia: {batch_accuracy:.4f}"
            )

    val_loss = val_loss / val_total
    val_accuracy = val_correct / val_total

    print(
        f"\nResumo da época {epoch + 1:02d} | "
        f"Loss treino: {train_loss:.4f} | "
        f"Acurácia treino: {train_accuracy:.4f} | "
        f"Loss validação: {val_loss:.4f} | "
        f"Acurácia validação: {val_accuracy:.4f}\n"
    )

Época 01 | Treino | Batch 1/25 | Loss: 0.6915 | Acurácia: 0.5312
Época 01 | Treino | Batch 2/25 | Loss: 0.5517 | Acurácia: 0.6992
Época 01 | Treino | Batch 3/25 | Loss: 0.4964 | Acurácia: 0.7578
Época 01 | Treino | Batch 4/25 | Loss: 0.3793 | Acurácia: 0.8477
Época 01 | Treino | Batch 5/25 | Loss: 0.3059 | Acurácia: 0.8633
Época 01 | Treino | Batch 6/25 | Loss: 0.3368 | Acurácia: 0.8242
Época 01 | Treino | Batch 7/25 | Loss: 0.2523 | Acurácia: 0.8750
Época 01 | Treino | Batch 8/25 | Loss: 0.2565 | Acurácia: 0.8711
Época 01 | Treino | Batch 9/25 | Loss: 0.2204 | Acurácia: 0.9023
Época 01 | Treino | Batch 10/25 | Loss: 0.2208 | Acurácia: 0.9141
Época 01 | Treino | Batch 11/25 | Loss: 0.1758 | Acurácia: 0.9297
Época 01 | Treino | Batch 12/25 | Loss: 0.2829 | Acurácia: 0.8828
Época 01 | Treino | Batch 13/25 | Loss: 0.1265 | Acurácia: 0.9414
Época 01 | Treino | Batch 14/25 | Loss: 0.1713 | Acurácia: 0.9375
Época 01 | Treino | Batch 15/25 | Loss: 0.1251 | Acurácia: 0.9375
Época 01 | Treino |